In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000096.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000261.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000114.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000097.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000019.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000085.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000006.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000153.txt
/kaggle/input/datasets/nandini5002/all-labeled-chandrayan/final_dataset/final_dataset/labels/img_000251.txt
/kaggle/input/datasets/nandi

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00


In [3]:
import os
import shutil
import random
import zipfile
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
WORK       = Path("/kaggle/working")
ALL_IMAGES = WORK / "all_images"
ALL_LABELS = WORK / "all_labels"
ALL_IMAGES.mkdir(parents=True, exist_ok=True)
ALL_LABELS.mkdir(parents=True, exist_ok=True)
 
BASE = Path("/kaggle/input/datasets/nandini5002/all-labeled-chandrayan")
 
SOURCES = [
    {"name": "final_dataset", "images": BASE / "final_dataset" / "final_dataset" / "images",
                               "labels": BASE / "final_dataset" / "final_dataset" / "labels"},
    {"name": "split_1",        "images": BASE / "split_1"       / "split_1"       / "images",
                               "labels": BASE / "split_1"       / "split_1"       / "labels"},
    {"name": "split_2",        "images": BASE / "split_2"       / "split_2"       / "images",
                               "labels": BASE / "split_2"       / "split_2"       / "labels"},
]
 
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
 
def safe_copy(src: Path, dest_dir: Path, tag: str):
    dest = dest_dir / src.name
    if dest.exists():                          # collision -> add dataset tag
        dest = dest_dir / f"{src.stem}__{tag}{src.suffix}"
    shutil.copy(src, dest)
 
print("=" * 52)
print("  Merging datasets")
print("=" * 52)
 
total_imgs = total_lbls = 0
 
for src in SOURCES:
    name       = src["name"]
    img_dir    = src["images"]
    lbl_dir    = src["labels"]
 
    img_paths  = [f for f in img_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS]
    lbl_lookup = {f.stem: f for f in lbl_dir.iterdir() if f.suffix == ".txt"}
 
    copied_i = copied_l = 0
    for img in img_paths:
        safe_copy(img, ALL_IMAGES, name)
        copied_i += 1
 
        lbl = lbl_lookup.get(img.stem)
        if lbl:
            safe_copy(lbl, ALL_LABELS, name)
            copied_l += 1
        else:
            (ALL_LABELS / f"{img.stem}.txt").touch()   # empty = background image
 
    print(f"  ✅ {name:20s} -> {copied_i:3d} images  |  {copied_l:3d} labels")
    total_imgs += copied_i
    total_lbls += copied_l
 
print(f"\n{'-'*52}")
print(f"  Total images : {total_imgs}  (expected ~850)")
print(f"  Total labels : {total_lbls}")
print(f"{'-'*52}")
 
assert total_imgs >= 840, f"Only {total_imgs} images found — check paths above!"
print("\n  ✅ Merge complete. Proceed to stratified split.")


  Merging datasets
  ✅ final_dataset        -> 262 images  |  262 labels
  ✅ split_1              -> 294 images  |  294 labels
  ✅ split_2              -> 294 images  |  294 labels

----------------------------------------------------
  Total images : 850  (expected ~850)
  Total labels : 850
----------------------------------------------------

  ✅ Merge complete. Proceed to stratified split.


In [5]:
def stratified_split(
    images_dir,
    labels_dir,
    output_dir,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Ratios must sum to 1"
    random.seed(seed)
 
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    output_dir = Path(output_dir)
 
    # Categorize each image by its highest-priority class
    # Priority: large(2) > medium(1) > small(0) > background(no labels)
    buckets = defaultdict(list)
    all_images = list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png"))
 
    for img_path in all_images:
        label_path = labels_dir / (img_path.stem + ".txt")
 
        if not label_path.exists() or label_path.stat().st_size == 0:
            buckets["background"].append(img_path.stem)
            continue
 
        classes_in_image = set()
        with open(label_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    classes_in_image.add(int(line.split()[0]))
 
        if 2 in classes_in_image:
            buckets["large"].append(img_path.stem)
        elif 1 in classes_in_image:
            buckets["medium"].append(img_path.stem)
        else:
            buckets["small"].append(img_path.stem)
 
    # Split each bucket separately (stratified)
    train_ids, val_ids, test_ids = [], [], []
 
    print("\nStratified split breakdown:")
    for bucket_name, stems in buckets.items():
        random.shuffle(stems)
        n = len(stems)
        n_val   = max(1, round(n * val_ratio))
        n_test  = max(1, round(n * test_ratio))
        n_train = n - n_val - n_test
 
        train_ids.extend(stems[:n_train])
        val_ids.extend(stems[n_train:n_train + n_val])
        test_ids.extend(stems[n_train + n_val:])
 
        print(f"  {bucket_name:12s}: {n} total → "
              f"train={n_train}, val={n_val}, test={n_test}")
 
    # Copy files into split folders
    for split_name, stems in [("train", train_ids),
                               ("val",   val_ids),
                               ("test",  test_ids)]:
        img_out = output_dir / "images" / split_name
        lbl_out = output_dir / "labels" / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
 
        for stem in stems:
            for ext in [".jpg", ".jpeg", ".png"]:
                src = images_dir / (stem + ext)
                if src.exists():
                    shutil.copy(src, img_out / src.name)
                    break
 
            lbl_src = labels_dir / (stem + ".txt")
            if lbl_src.exists():
                shutil.copy(lbl_src, lbl_out / (stem + ".txt"))
            else:
                (lbl_out / (stem + ".txt")).touch()  # empty file for background
 
    # Summary
    print(f"\n{'─'*45}")
    print(f"  Total images : {len(all_images)}")
    print(f"  Train        : {len(train_ids)}")
    print(f"  Val          : {len(val_ids)}")
    print(f"  Test         : {len(test_ids)}")
    print(f"{'─'*45}")
 
    # Write data.yaml
    yaml_content = f"""path: {output_dir.resolve()}
train: images/train
val:   images/val
test:  images/test
 
nc: 3
names: ["crater_small", "crater_medium", "crater_large"]
"""
    with open(output_dir / "data.yaml", "w") as f:
        f.write(yaml_content)
 
    print(f"  data.yaml → {output_dir / 'data.yaml'}")
    return output_dir / "data.yaml"
 
 
yaml_path = stratified_split(
    images_dir  = ALL_IMAGES,
    labels_dir  = ALL_LABELS,
    output_dir  = WORK / "dataset_final",
    train_ratio = 0.70,
    val_ratio   = 0.15,
    test_ratio  = 0.15,
    seed        = 42
)


Stratified split breakdown:
  small       : 211 total → train=147, val=32, test=32
  medium      : 194 total → train=136, val=29, test=29
  large       : 391 total → train=273, val=59, test=59
  background  : 54 total → train=38, val=8, test=8

─────────────────────────────────────────────
  Total images : 850
  Train        : 594
  Val          : 128
  Test         : 128
─────────────────────────────────────────────
  data.yaml → /kaggle/working/dataset_final/data.yaml


In [6]:
model = YOLO("yolo11s.pt")   # Small model — 850 images justifies it
 
results = model.train(
    data            = str(yaml_path),
    epochs          = 150,
    imgsz           = 640,        # native resolution
    batch           = 16,         # safe for P100/T4 at 640px
    patience        = 30,
    lr0             = 0.001,
    lrf             = 0.01,
    warmup_epochs   = 5,
    warmup_bias_lr  = 0.05,
    flipud          = 0.5,
    fliplr          = 0.5,
    degrees         = 45.0,
    mosaic          = 1.0,
    close_mosaic    = 30,
    copy_paste      = 0.3,
    cls             = 0.5,
    save_period     = 10,
    val             = True,
)
 
shutil.copy(
    os.path.join(results.save_dir, "weights/best.pt"),
    str(WORK / "best_final.pt")
)
print("✅ Saved best_final.pt")
 

 

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=30, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_final/data.yaml, degrees=45.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

In [7]:
model = YOLO(str(WORK / "best_final.pt"))
 
metrics = model.val(
    data   = str(yaml_path),
    split  = "test",
    imgsz  = 640,
)
 
print(f"\nTest mAP50:      {metrics.box.map50:.3f}")
print(f"Test mAP50-95:   {metrics.box.map:.3f}")
print(f"Per-class mAP50: {metrics.box.maps}")
# Index: [0]=crater_small  [1]=crater_medium  [2]=crater_large
 

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,961 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2030.8±302.7 MB/s, size: 197.5 KB)
val: Scanning /kaggle/working/dataset_final/labels/test... 128 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 1.0Kit/s 0.1s
val: New cache created: /kaggle/working/dataset_final/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 12% ━─────────── 1/8 1.4s/it 0.4s<9.5sWARNING ⚠️ NMS time limit 2.800s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.1it/s 7.5s
                   all        128       3405      0.604      0.578      0.635      0.407
          crater_small        116       3147      0.559      0.476      0.541      0.282
         crater_medium         73        160      0.546      0.544

In [8]:
out_zip = WORK / "final_model.zip"
 
with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Best weights
    zf.write(WORK / "best_final.pt", "best_final.pt")
    # Full training run (curves, confusion matrix, etc.)
    for f in Path(results.save_dir).rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(WORK))
 
print(f"✅ final_model.zip ready — download from Kaggle output panel")
print(f"   Size: {out_zip.stat().st_size / 1e6:.1f} MB")


✅ final_model.zip ready — download from Kaggle output panel
   Size: 1074.6 MB
